# 2번 담당자 실행용 Colab — Decision Tree · Random Forest · ExtraTrees

**발표일: 2026-12-02 | 담당: 트리 모델 개발 + 최종 발표**

- **11/19**: 1번에게 공통 분할 ZIP 수령 → B~D 실행
- **11/20~21**: E 기본 모델(교차검증)
- **11/22~24**: F 최적화(교차검증 기반)
- **11/25**: G `rf_best_params.json` + `feature_importance.csv`를 **1번에게 먼저 공유**
- **11/26**: H~J 최종 테스트(한 번) + 예측 결과를 4번에게 공유

**사용법**: [Google Colab](https://colab.research.google.com/) → 파일 → 노트 업로드 → 이 `.ipynb` 선택 → 위에서부터 차례로 실행.

**절대 하지 말 것**: 데이터를 임의로 다시 80:20 분할하거나, `quality`/`sample_id`를 입력 X에 포함하거나, 최종 Test 성능을 보고 튜닝하는 행동.

## A. 실행 환경 · 저장 폴더
Colab 기본 CPU에서 실행 가능합니다. 이 노트북은 NumPy, Pandas, Scikit-learn, Matplotlib 등 일반 패키지만 사용합니다. 코드 실행 결과는 Colab `/content/AI_Project/`에 만들어지므로 마지막에 ZIP을 내려받아 공용 Drive로 옮겨 주세요.

In [ ]:
import os, json, io, warnings, shutil, zipfile
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import (
    StratifiedKFold, StratifiedGroupKFold, cross_validate, RandomizedSearchCV
)
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier
from sklearn.metrics import (
    accuracy_score, f1_score, precision_score, recall_score, make_scorer,
    confusion_matrix, classification_report
)
import joblib
import sklearn
from IPython.display import display

SEED = 42
ROOT = Path(os.environ.get('PROJECT_OUTPUT_DIR', '/content/AI_Project'))
DATA = ROOT / 'data'
RESULTS = ROOT / 'results'
FIGURES = RESULTS / 'figures'
MODELS = RESULTS / 'models'
for p in (DATA,RESULTS,FIGURES,MODELS): p.mkdir(parents=True, exist_ok=True)
print('Scikit-learn 버전:', sklearn.__version__)
print('작업 폴더:', ROOT)

## B. 1번의 공통 분할 ZIP 업로드 · 파일 검사

1번에게 **`01_공통분할_즉시공유.zip`**을 받아 이 셀에서 업로드하세요. ZIP 내부는 `train_indices.csv`, `test_indices.csv`, `feature_columns.json`, `split_manifest.json`, `group_ids.csv`입니다.

- Colab에서는 코드 실행 중 업로드 버튼이 나타납니다.
- 로컬 테스트에서는 `PROJECT_SPLIT_ZIP` 환경변수에 ZIP 경로를 지정할 수 있습니다.
- **1번의 ZIP 없이 모델 학습을 시작하지 않습니다.**

In [ ]:
required = {
    'train_indices.csv', 'test_indices.csv', 'feature_columns.json',
    'split_manifest.json', 'group_ids.csv'
}
zip_env = os.environ.get('PROJECT_SPLIT_ZIP')
zip_path = Path(zip_env) if zip_env else ROOT / '01_공통분할_즉시공유.zip'
if not zip_path.exists():
    print('1번 담당자에게 받은 01_공통분할_즉시공유.zip 파일을 업로드하세요.')
    try:
        from google.colab import files
        uploaded = files.upload()
    except ImportError as e:
        raise FileNotFoundError('Colab이 아닙니다. PROJECT_SPLIT_ZIP에 ZIP 경로를 지정하세요.') from e
    if not uploaded: raise RuntimeError('업로드된 ZIP이 없습니다.')
    zip_path = Path(next(iter(uploaded)))
with zipfile.ZipFile(zip_path) as z:
    names = {Path(n).name: n for n in z.namelist() if not n.endswith('/')}
    missing = required - set(names)
    if missing:
        raise ValueError('분할 ZIP 필수 파일이 없습니다: '+str(sorted(missing)))
    for name in sorted(required):
        (DATA/name).write_bytes(z.read(names[name]))
print('공통 분할 파일 읽기 완료:')
for p in sorted(DATA.iterdir()): print(' -',p.name, p.stat().st_size,'bytes')
manifest = json.loads((DATA/'split_manifest.json').read_text(encoding='utf-8'))
features = json.loads((DATA/'feature_columns.json').read_text(encoding='utf-8'))
assert len(features) == 13 and 'quality' not in features
print('1번의 분할 방법:',manifest['split_method'])
print('그룹 기반 CV 필요:',manifest['grouped_cv_required'])

## C. GitHub 원본 `data.csv` 불러오기

1번과 **완전히 동일한 원본 CSV**를 사용해야 합니다. 새로 데이터를 정렬·삭제하면 행 번호가 바뀝니다. CSV는 `sep=";"`으로 읽습니다. GitHub 연결이 실패하면 [원본 CSV 링크](https://github.com/airtlab/machine-learning-for-quality-prediction-in-plastic-injection-molding/blob/master/dataset/data.csv)에서 내려받아 업로드하세요.

In [ ]:
RAW_URL = (
  'https://raw.githubusercontent.com/airtlab/'
  'machine-learning-for-quality-prediction-in-plastic-injection-molding/'
  'master/dataset/data.csv'
)
source = os.environ.get('PROJECT_CSV_PATH', RAW_URL)
try:
    raw = pd.read_csv(source, sep=';', encoding='utf-8-sig')
except Exception as e:
    print('온라인 CSV 로딩 실패:',str(e)[:250])
    print('원본 dataset/data.csv를 업로드하세요.')
    try:
        from google.colab import files
        uploaded = files.upload()
    except ImportError as e2:
        raise RuntimeError('PROJECT_CSV_PATH에 로컬 CSV 경로를 지정하세요.') from e2
    if not uploaded: raise RuntimeError('CSV를 업로드하지 않았습니다.')
    raw = pd.read_csv(next(iter(uploaded)), sep=';', encoding='utf-8-sig')
raw.columns = raw.columns.astype(str).str.strip()
if raw.columns.tolist() != features + ['quality']:
    raise ValueError('1번의 feature_columns.json과 원본 CSV 열 이름/순서가 다릅니다. '                     '원본 데이터인지 또는 sep=";"인지 확인하세요.\n'+str(raw.columns.tolist()))
if len(raw) != manifest['total_rows']:
    raise ValueError(f"1번 분할 당시 {manifest['total_rows']}행이었는데 지금 {len(raw)}행입니다. 원본 CSV를 다시 확인하세요.")
df = raw.copy()
df.insert(0, 'sample_id', np.arange(len(df), dtype=int))
for col in features:
    df[col] = pd.to_numeric(df[col], errors='coerce')
y_num = pd.to_numeric(df['quality'], errors='coerce')
if y_num.isna().any() or not y_num.isin([1,2,3,4]).all():
    raise ValueError('quality 라벨이 1~4 중 하나가 아닙니다.')
df['quality'] = y_num.astype(int)
print('확인:',len(df),'개 샘플,',len(features),'개 입력변수')
print('등급 분포:\n',df['quality'].value_counts().sort_index().to_string())
display(df.head(3))

## D. **1번과 완전히 같은** 학습·시험 행 선택

`sample_id`를 원본 행 번호(0부터 시작)로 해석합니다. 여기서는 별도로 분할하지 않습니다. 입력이 중복된 경우 1번의 `group_ids.csv`를 이용해 그룹이 학습/시험에 동시에 존재하지 않는지 검사합니다.

In [ ]:
train_ids = pd.read_csv(DATA/'train_indices.csv')['sample_id'].astype(int).to_numpy()
test_ids  = pd.read_csv(DATA/'test_indices.csv')['sample_id'].astype(int).to_numpy()
assert len(np.unique(train_ids)) == len(train_ids), '학습 sample_id 중복'
assert len(np.unique(test_ids)) == len(test_ids), '시험 sample_id 중복'
assert len(set(train_ids)&set(test_ids)) == 0, '학습/시험 행 번호 중복!'
assert set(train_ids)|set(test_ids) == set(df['sample_id']), '일부 행 누락 또는 잘못된 id'
assert len(train_ids) == manifest['train_rows'] and len(test_ids) == manifest['test_rows']
group_df = pd.read_csv(DATA/'group_ids.csv')
assert set(group_df.columns) == {'sample_id','group_id'}
assert len(group_df)==len(df) and group_df['sample_id'].is_unique
ordered = df.set_index('sample_id', drop=False)
train_df = ordered.loc[train_ids].copy()
test_df  = ordered.loc[test_ids].copy()
X_train = train_df[features].copy()
y_train = train_df['quality'].copy()
X_test  = test_df[features].copy()
y_test  = test_df['quality'].copy()
group_series = group_df.set_index('sample_id')['group_id']
train_groups = group_series.loc[train_ids].to_numpy()
test_groups = group_series.loc[test_ids].to_numpy()
use_group_cv = bool(manifest['grouped_cv_required'])
if use_group_cv:
    assert len(set(train_groups)&set(test_groups))==0, '같은 공정 입력 그룹이 Train/Test에 섞였습니다!'
    cv = StratifiedGroupKFold(n_splits=5,shuffle=True,random_state=SEED)
    print('StratifiedGroupKFold 사용: 그룹 중복 방지')
else:
    cv = StratifiedKFold(n_splits=5,shuffle=True,random_state=SEED)
    print('StratifiedKFold 사용')
cv_params = {'groups':train_groups} if use_group_cv else {}
print('학습·검증:',len(train_df),'행 / 테스트:',len(test_df),'행')
print('입력 누락값:',int(X_train.isna().sum().sum()))
print('학습 등급:',y_train.value_counts().sort_index().to_dict())
print('테스트 등급:',y_test.value_counts().sort_index().to_dict())
assert list(X_train.columns)==features and 'quality' not in X_train.columns

## E. 기본 모델 3개 성능 측정 — 11/20~21

Decision Tree, Random Forest, ExtraTrees를 동일한 **학습 데이터 내부 5-Fold 교차검증**으로 평가합니다. `SimpleImputer`는 결측치가 있어도 학습 폴드 안에서 중앙값을 추정합니다. 테스트 데이터는 여기서 **아예 사용하지 않습니다**.

- Accuracy: 전체 정답 비율
- Macro F1: 4등급을 동등하게 고려한 F1 평균
- Waste Recall: 실제 1등급 중 1등급으로 맞춘 비율

각 점수는 0~1이며, 0.95는 95%를 뜻합니다.

In [ ]:
def waste_recall_metric(y_true,y_pred):
    return recall_score(y_true,y_pred,labels=[1],average='macro',zero_division=0)
SCORING = {
    'accuracy':'accuracy',
    'macro_f1':'f1_macro',
    'waste_recall':make_scorer(waste_recall_metric)
}
def pipe(model):
    return Pipeline([
        ('imputer',SimpleImputer(strategy='median')),
        ('model',model)
    ])
baselines = {
    'DecisionTree': pipe(DecisionTreeClassifier(random_state=SEED)),
    'RandomForest': pipe(RandomForestClassifier(n_estimators=120,random_state=SEED,n_jobs=1)),
    'ExtraTrees': pipe(ExtraTreesClassifier(n_estimators=120,random_state=SEED,n_jobs=1)),
}
baseline_rows=[]
for name,model in baselines.items():
    scores = cross_validate(model, X_train,y_train,cv=cv,groups=cv_params.get('groups'),
                            scoring=SCORING,n_jobs=-1,error_score='raise')
    row={'model':name}
    for metric in SCORING:
        arr=scores['test_'+metric]
        row['cv_'+metric+'_mean']=float(arr.mean())
        row['cv_'+metric+'_std']=float(arr.std())
    baseline_rows.append(row)
    print(name,'CV macro F1:',round(row['cv_macro_f1_mean'],4),
          'Accuracy:',round(row['cv_accuracy_mean'],4),
          'Waste Recall:',round(row['cv_waste_recall_mean'],4))
baseline_table = pd.DataFrame(baseline_rows)
baseline_table.to_csv(RESULTS/'cv_baseline_metrics.csv',index=False,encoding='utf-8-sig')
display(baseline_table)

## F. 하이퍼파라미터 튜닝 — 11/22~24

각 모델에 `RandomizedSearchCV`로 8개 조합(Decision Tree는 10개)을 시도합니다. 선정 기준은 **CV Macro F1**입니다. 탐색 중 발생하는 CV 점수는 하이퍼파라미터 선택에도 사용되므로 엄밀한 독립 검증 점수가 아닙니다. 최종 Test는 H에서 한 번만 측정합니다.

- 실행이 너무 오래 걸린다면 아래 `N_ITER` 값을 8→4처럼 줄여도 됩니다. 수정 사실을 발표 자료에 기록하세요.
- 원논문은 RapidMiner의 ExtraTrees 계열을 RF로 다뤘으므로, 이번에는 Scikit-learn RF와 ExtraTrees를 별도로 비교합니다.
- 튜닝 전에 최종 Test 점수를 조회하지 않습니다.

In [ ]:
PARAMS = {
    'DecisionTree': {
        'model__max_depth':[None,4,8,12,20],
        'model__min_samples_leaf':[1,2,4,8],
        'model__min_samples_split':[2,5,10]
    },
    'RandomForest': {
        'model__n_estimators':[100,160,250],
        'model__max_depth':[None,8,16,24],
        'model__min_samples_leaf':[1,2,4],
        'model__max_features':['sqrt',0.8]
    },
    'ExtraTrees': {
        'model__n_estimators':[100,160,250],
        'model__max_depth':[None,8,16,24],
        'model__min_samples_leaf':[1,2,4],
        'model__max_features':['sqrt',0.8]
    }
}
N_ITER = {'DecisionTree':10, 'RandomForest':8, 'ExtraTrees':8}
searches={}
tuned_rows=[]
for name,base in baselines.items():
    print('\n튜닝 중:',name)
    search=RandomizedSearchCV(
       estimator=base,
       param_distributions=PARAMS[name],
       n_iter=N_ITER[name],
       scoring=SCORING,
       refit='macro_f1',
       cv=cv,
       random_state=SEED,
       n_jobs=-1,
       error_score='raise',
       return_train_score=False
    )
    search.fit(X_train,y_train,**cv_params)
    searches[name]=search
    # best_index_ 의 동일 후보에서 Accuracy와 Waste Recall도 함께 가져옵니다.
    idx=search.best_index_
    row={
        'model':name,
        'tuned_cv_macro_f1_mean':float(search.cv_results_['mean_test_macro_f1'][idx]),
        'tuned_cv_macro_f1_std':float(search.cv_results_['std_test_macro_f1'][idx]),
        'tuned_cv_accuracy_mean':float(search.cv_results_['mean_test_accuracy'][idx]),
        'tuned_cv_waste_recall_mean':float(search.cv_results_['mean_test_waste_recall'][idx]),
        'best_parameters':json.dumps(search.best_params_,ensure_ascii=False),
        'n_candidates':len(search.cv_results_['params'])
    }
    tuned_rows.append(row)
    pd.DataFrame(search.cv_results_).to_csv(
       RESULTS/f'cv_search_results_{name}.csv',index=False,encoding='utf-8-sig')
    print('CV macro F1=',round(row['tuned_cv_macro_f1_mean'],4),
          '/ CV accuracy=',round(row['tuned_cv_accuracy_mean'],4))
    print('선택된 파라미터:',search.best_params_)
tuned_table=pd.DataFrame(tuned_rows)
tuned_table.to_csv(RESULTS/'cv_tuning_summary.csv',index=False,encoding='utf-8-sig')
display(tuned_table[['model','tuned_cv_macro_f1_mean','tuned_cv_accuracy_mean',
                     'tuned_cv_waste_recall_mean','n_candidates']])

## G. 1번에게 **먼저** 보내는 변수 중요도 결과 — 11/25

튜닝이 완료된 Random Forest를 전체 **학습 데이터에만** 적합시켜 `feature_importances_`를 구합니다. **Test를 사용할 필요가 없습니다.**

1번은 `rf_best_params.json`을 받아 자신의 변수 제거 실험을 합니다. 파라미터는 `model__` 접두어를 제거하여 Scikit-learn `RandomForestClassifier(**params)`에 직접 들어갈 형식입니다.

1번에게 보낼 파일:
- `rf_best_params.json`
- `feature_importance.csv` (열: `feature`, `importance`)

`02_1번에게_먼저_공유.zip`을 내려받아 공용 Drive에 올리면 됩니다.

In [ ]:
rf_search=searches['RandomForest']
rf_pipe=rf_search.best_estimator_   # refit=True이므로 전체 학습 데이터로 이미 적합됨
rf_params={key.replace('model__',''): value for key,value in rf_search.best_params_.items()}
rf_params['random_state']=SEED
rf_params['n_jobs']=1
with (DATA/'rf_best_params.json').open('w',encoding='utf-8') as f:
    json.dump(rf_params,f,ensure_ascii=False,indent=2)
importance=pd.DataFrame({
    'feature':features,
    'importance':rf_pipe.named_steps['model'].feature_importances_
}).sort_values('importance',ascending=False,ignore_index=True)
importance.to_csv(RESULTS/'feature_importance.csv',index=False,encoding='utf-8-sig')
plt.figure(figsize=(11,6))
plt.barh(importance['feature'].iloc[::-1],importance['importance'].iloc[::-1])
plt.xlabel('Random Forest impurity-based feature importance')
plt.title('Training data only: Random Forest feature importance')
plt.tight_layout()
plt.savefig(FIGURES/'feature_importance_rf.png',dpi=160,bbox_inches='tight')
plt.close()
display(importance)
share_path=ROOT/'02_1번에게_먼저_공유.zip'
with zipfile.ZipFile(share_path,'w',compression=zipfile.ZIP_DEFLATED) as z:
    z.write(DATA/'rf_best_params.json',arcname='rf_best_params.json')
    z.write(RESULTS/'feature_importance.csv',arcname='feature_importance.csv')
    z.write(FIGURES/'feature_importance_rf.png',arcname='feature_importance_rf.png')
print('1번에게 즉시 보낼 파일:',share_path)
try:
    from google.colab import files
    files.download(str(share_path))
except ImportError:
    print('Colab 이외 실행 환경: 위 경로에서 파일 확인')

## H. 최종 모델 선정 + **단 한 번의** 최종 테스트 — 11/26

**반드시 F(튜닝)와 G(변수 중요도) 이후에 실행하세요.**

- **우리 팀이 추천하는 대표 알고리즘은 CV Macro F1이 가장 높은 모델**로 먼저 정합니다. Test 정확도를 보고 대표 모델을 바꾸면 안 됩니다.
- 모든 모델의 Test 성능은 최종 보고용으로 한 번 계산해 공유합니다.
- 교차검증 선택 점수와 테스트 점수는 서로 다른 열로 표기합니다.

`tree_predictions.csv`에는 `sample_id`, `model`, `y_true`, `y_pred`, 그리고 `prob_1`~`prob_4`를 저장하여 4번 담당자가 혼동행렬과 오분류 원인을 분석할 수 있게 합니다.

In [ ]:
ranked=tuned_table.sort_values(
    ['tuned_cv_macro_f1_mean','tuned_cv_waste_recall_mean','tuned_cv_accuracy_mean'],
    ascending=[False,False,False]
).reset_index(drop=True)
selected_model=ranked.loc[0,'model']
(DATA/'selected_tree_model.json').write_text(
    json.dumps({'selected_by':'training CV macro F1 (tie: Waste Recall, Accuracy)',
                'selected_model':selected_model,'seed':SEED},ensure_ascii=False,indent=2),
    encoding='utf-8')
print('최종 Test 전에 CV로 선정한 대표 트리 모델:',selected_model)
# 이 셀을 반복 실행하더라도 다른 하이퍼파라미터를 선택하지 마세요.
metrics_rows=[]
prediction_parts=[]
for name in ['DecisionTree','RandomForest','ExtraTrees']:
    model=searches[name].best_estimator_
    y_pred=model.predict(X_test)
    probs=model.predict_proba(X_test)
    class_ids=model.named_steps['model'].classes_
    pred_df=pd.DataFrame({
        'sample_id':test_df['sample_id'].to_numpy(),
        'model':name,
        'y_true':y_test.to_numpy(),
        'y_pred':y_pred.astype(int)
    })
    for k in [1,2,3,4]:
        pred_df[f'prob_{k}']=probs[:,list(class_ids).index(k)] if k in class_ids else 0.0
    prediction_parts.append(pred_df)
    sel = tuned_table.set_index('model').loc[name]
    metrics_rows.append({
        'model':name,
        'cv_macro_f1_mean_tuned_selection':sel['tuned_cv_macro_f1_mean'],
        'cv_accuracy_mean_tuned_selection':sel['tuned_cv_accuracy_mean'],
        'cv_waste_recall_mean_tuned_selection':sel['tuned_cv_waste_recall_mean'],
        'test_accuracy':accuracy_score(y_test,y_pred),
        'test_macro_f1':f1_score(y_test,y_pred,average='macro',zero_division=0),
        'test_macro_precision':precision_score(y_test,y_pred,average='macro',zero_division=0),
        'test_macro_recall':recall_score(y_test,y_pred,average='macro',zero_division=0),
        'test_waste_recall':recall_score(y_test,y_pred,labels=[1],average='macro',zero_division=0),
        'false_acceptance_count':int(((y_test.to_numpy()==1)&(y_pred!=1)).sum()),
        'selected_as_tree_representative':name==selected_model
    })
    joblib.dump(model,MODELS/f'{name}_trained.joblib')
    print(name,'최종 Test 정확도:',round(metrics_rows[-1]['test_accuracy'],4),
          'Macro F1:',round(metrics_rows[-1]['test_macro_f1'],4))
metrics=pd.DataFrame(metrics_rows)
predictions=pd.concat(prediction_parts,ignore_index=True)
metrics.to_csv(RESULTS/'metrics.csv',index=False,encoding='utf-8-sig')
predictions.to_csv(RESULTS/'tree_predictions.csv',index=False,encoding='utf-8-sig')
assert len(predictions)==len(test_df)*3
assert not predictions[['sample_id','model']].duplicated().any()
assert predictions['y_true'].isin([1,2,3,4]).all()
assert (predictions[[f'prob_{k}' for k in range(1,5)]].sum(axis=1).round(5)==1).all()
display(metrics)
display(predictions.head())

## I. 2번 담당자 분석 그래프 / 발표용 메모 — 11/26 이후

- CV 기본 모델과 튜닝 모델 점수를 비교해 성능 개선이 있었는지 확인합니다. 단, 튜닝에 사용된 CV 점수는 낙관적으로 추정될 수 있습니다.
- **최종 Test 정확도는 따로 표시**합니다. 선정 기준이 아니며 최종 평가 결과입니다.
- 변수 중요도가 높아도 *인과관계*를 뜻하지 않습니다.
- 원논문의 95.04%는 저자들의 RapidMiner 기반 5-Fold CV 점수이므로, 우리 팀의 20% 고정 Test 점수와 직접적인 우열 비교를 하지 않습니다.

In [ ]:
plt.figure(figsize=(8,4))
merged=baseline_table.merge(tuned_table,on='model')
x=np.arange(len(merged)); w=0.36
plt.bar(x-w/2,merged['cv_macro_f1_mean'],w,label='Baseline CV F1')
plt.bar(x+w/2,merged['tuned_cv_macro_f1_mean'],w,label='Tuned selection CV F1')
plt.xticks(x,merged['model']);plt.ylim(0,1);plt.ylabel('Macro F1')
plt.title('Train-only CV: baseline vs tuning')
plt.legend();plt.tight_layout()
plt.savefig(FIGURES/'cv_before_after_f1.png',dpi=170,bbox_inches='tight');plt.close()
plt.figure(figsize=(8,4))
plt.bar(metrics['model'],metrics['test_accuracy'])
plt.ylim(0,1);plt.ylabel('Test accuracy')
plt.title('Held-out Test accuracy (final reporting only)')
plt.tight_layout()
plt.savefig(FIGURES/'final_test_accuracy.png',dpi=170,bbox_inches='tight');plt.close()
report=[]
report.append('# 2번 담당 트리 모델 결과 정리')
report.append('')
report.append(f'- 공통 학습 데이터: {len(train_df)}개 / 공통 테스트 데이터: {len(test_df)}개')
report.append(f'- 분할 방식: {manifest["split_method"]}')
report.append(f'- 내부 검증: {type(cv).__name__}, 5 folds')
report.append(f'- CV Macro F1 기준 대표 트리 모델: **{selected_model}**')
report.append('')
report.append('## 튜닝 CV 결과 (학습 데이터 내 모델 선택 점수)')
for _,r in tuned_table.iterrows():
    report.append(f'- {r["model"]}: macro F1={r["tuned_cv_macro_f1_mean"]:.4f}, accuracy={r["tuned_cv_accuracy_mean"]:.4f}')
report.append('')
report.append('## 최종 테스트 결과 (모델 선정에 사용하지 않음)')
for _,r in metrics.iterrows():
    report.append(f'- {r["model"]}: accuracy={r["test_accuracy"]:.4f}, macro F1={r["test_macro_f1"]:.4f}, Waste Recall={r["test_waste_recall"]:.4f}')
report.append('')
report.append('## 중요 변수 (RF 학습 기반)')
for _,r in importance.head(5).iterrows():
    report.append(f'- {r["feature"]}: importance={r["importance"]:.4f}')
report.append('')
report.append('## 보고 시 주의')
report.append('- 이 점수는 Colab 실행 결과이며 원논문에 기재된 점수와 다른 평가 설계임.')
report.append('- 테스트 점수를 보고 하이퍼파라미터를 다시 선택하지 않음.')
report.append('- 중요도는 예측 활용도이지 공정변수의 인과적 영향 크기가 아님.')
(RESULTS/'02_모델분석_초안.md').write_text('\n'.join(report),encoding='utf-8')
print('\n'.join(report[:20]))

## J. **4번에게** 최종 결과 ZIP 전달 — 11/26

결과 ZIP `02_트리모델_최종공유.zip`에는 `metrics.csv`, `tree_predictions.csv`, 모든 튜닝 결과, 모델 `.joblib`, 그래프, `rf_best_params.json` 등이 들어 있습니다.

공용 Drive의 `results`에 올리고 4번에게 파일 경로를 알려 주세요. 1번에게 전달한 `rf_best_params.json`과 동일한 설정인지 확인합니다.

In [ ]:
final_zip=ROOT/'02_트리모델_최종공유.zip'
with zipfile.ZipFile(final_zip,'w',compression=zipfile.ZIP_DEFLATED) as z:
    for path in sorted(DATA.rglob('*')):
        if path.is_file() and path.name in {'rf_best_params.json','selected_tree_model.json','split_manifest.json'}:
            z.write(path,arcname=str(path.relative_to(ROOT)))
    for path in sorted(RESULTS.rglob('*')):
        if path.is_file(): z.write(path,arcname=str(path.relative_to(ROOT)))
print('최종 공유 ZIP:',final_zip)
print('반드시 포함:',RESULTS/'metrics.csv',RESULTS/'tree_predictions.csv')
print('총 파일 개수:',len(zipfile.ZipFile(final_zip).namelist()))
try:
    from google.colab import files
    files.download(str(final_zip))
except ImportError:
    print('Colab 이외 환경: 위 경로 파일 확인')